<a href="https://colab.research.google.com/github/xyzplanet/RFM_project/blob/main/RFM_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!wget https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv -O online_retail.csv

--2026-09-22 21:51:15--  https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 45038760 (43M) [text/plain]
Saving to: ‘online_retail.csv’

online_retail.csv   100%[===================>]  42.95M  --.-KB/s    in 0.09s   

2026-09-22 21:51:16 (456 MB/s) - ‘online_retail.csv’ saved [45038760/45038760]



In [20]:
import pandas as pd
# read the dataset
df = pd.read_csv('online_retail.csv')
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [21]:
print(df.shape)
print(df.dtypes)
df.isnull().sum()




(541909, 8)
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object


,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


In [22]:
# clean data
import numpy as np

print(f"Initial raw rows: {len(df)}")

# Step 1: Remove Duplicate Rows
df.drop_duplicates(inplace=True)

# Step 2: Handle the rows with missing values
df.dropna(subset=['CustomerID'], inplace=True)

# Step 3: Data Type Casting
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Step 4: Filter Returns and Anomalies
# Filter out cancelled orders
df_clean = df[~df['InvoiceNo'].str.startswith('C', na=False)].copy()
# Filter out zero or negative Quantity and UnitPrice
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

# Step 5: Feature Engineering
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']
# Extract date dimensions for time-series analysis
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')
df_clean['Date'] = df_clean['InvoiceDate'].dt.date

# Step 6: Final Verification
df_clean = df_clean.reset_index(drop=True)
print(f"Cleaned dataset rows: {len(df_clean)}")
print("\n--- 清洗后数据概览 ---")
print(df_clean.info())

Initial raw rows: 541909
Cleaned dataset rows: 392692

--- 清洗后数据概览 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392692 entries, 0 to 392691
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392692 non-null  object        
 1   StockCode    392692 non-null  object        
 2   Description  392692 non-null  object        
 3   Quantity     392692 non-null  int64         
 4   InvoiceDate  392692 non-null  datetime64[ns]
 5   UnitPrice    392692 non-null  float64       
 6   CustomerID   392692 non-null  object        
 7   Country      392692 non-null  object        
 8   TotalAmount  392692 non-null  float64       
 9   YearMonth    392692 non-null  period[M]     
 10  Date         392692 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(6), period[M](1)
memory usage: 33.0+ MB
None


In [23]:
df_clean['TotalSum'] = df_clean['Quantity'] * df_clean['UnitPrice']

In [24]:
print(df_clean.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392692 entries, 0 to 392691
Data columns (total 12 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392692 non-null  object        
 1   StockCode    392692 non-null  object        
 2   Description  392692 non-null  object        
 3   Quantity     392692 non-null  int64         
 4   InvoiceDate  392692 non-null  datetime64[ns]
 5   UnitPrice    392692 non-null  float64       
 6   CustomerID   392692 non-null  object        
 7   Country      392692 non-null  object        
 8   TotalAmount  392692 non-null  float64       
 9   YearMonth    392692 non-null  period[M]     
 10  Date         392692 non-null  object        
 11  TotalSum     392692 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(6), period[M](1)
memory usage: 36.0+ MB
None


In [25]:
df_clean.to_csv('online_retail_clean.csv', index=False)

In [9]:
import sqlite3

df_clean['YearMonth'] = df_clean['YearMonth'].astype(str)
conn = sqlite3.connect('retail_analytics.db')
df_clean.to_sql('stg_fact_sales', conn, if_exists='replace', index=False)

392692

In [10]:
!pip install ipython-sql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 83.4 MB/s eta 0:00:00


In [11]:
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%sql sqlite:///retail_analytics.db

In [33]:
%%sql rfm_result <<
WITH Customer_Base AS (
    SELECT
        CustomerID,
        JULIANDAY('2011-12-10') - JULIANDAY(MAX(InvoiceDate)) AS Recency,
        COUNT(DISTINCT InvoiceNo) AS Frequency,
        SUM(TotalSum) AS Monetary
    FROM stg_fact_sales
    GROUP BY CustomerID
)
SELECT
    CustomerID,
    NTILE(5) OVER (ORDER BY Recency DESC) AS R_Score,
    NTILE(5) OVER (ORDER BY Frequency ASC) AS F_Score,
    NTILE(5) OVER (ORDER BY Monetary ASC) AS M_Score
FROM Customer_Base
LIMIT 10;

 * sqlite:///retail_analytics.db
Done.
Returning data to local variable rfm_result


In [14]:
%%sql rfm_result <<
WITH Base_Data AS (
    SELECT
        CustomerID,
        InvoiceNo,
        TotalAmount,
        -- Convert ISO date string to UNIX timestamp (seconds)
        strftime('%s', InvoiceDate) AS InvoiceTimestamp
    FROM stg_fact_sales
),
Max_Date AS (
    -- Retrieve the latest transaction timestamp across the entire dataset as the anchor date
    SELECT MAX(InvoiceTimestamp) AS MaxTimestamp FROM Base_Data
),
Customer_RFM AS (
    SELECT
        b.CustomerID,
        -- Recency: Days since last purchase relative to the dataset snapshot date
        CAST((m.MaxTimestamp - MAX(b.InvoiceTimestamp)) / 86400.0 AS INTEGER) AS Recency,
        -- Frequency: Count of unique order IDs (InvoiceNo)
        COUNT(DISTINCT b.InvoiceNo) AS Frequency,
        -- Monetary: Total monetary spend per customer
        ROUND(SUM(b.TotalAmount), 2) AS Monetary
    FROM Base_Data b
    CROSS JOIN Max_Date m
    GROUP BY b.CustomerID
),
RFM_Scores AS (
    SELECT
        CustomerID,
        Recency,
        Frequency,
        Monetary,
        -- NTILE(5) Scoring (1 to 5)
        -- Recency: Lower is better (more recent), so order DESC (smaller days get higher score)
        NTILE(5) OVER (ORDER BY Recency DESC) AS R_Score,
        -- Frequency & Monetary: Higher is better, so order ASC
        NTILE(5) OVER (ORDER BY Frequency ASC) AS F_Score,
        NTILE(5) OVER (ORDER BY Monetary ASC) AS M_Score
    FROM Customer_RFM
)
SELECT
    CustomerID,
    Recency,
    Frequency,
    Monetary,
    R_Score,
    F_Score,
    M_Score,
    -- Concatenate scores into a 3-digit RFM segment string (e.g., '555', '111')
    (CAST(R_Score AS TEXT) || CAST(F_Score AS TEXT) || CAST(M_Score AS TEXT)) AS RFM_Segment
FROM RFM_Scores
ORDER BY Monetary DESC;

 * sqlite:///retail_analytics.db
Done.
Returning data to local variable rfm_result


In [15]:
rfm_df = rfm_result.DataFrame()
rfm_df.to_csv('rfm_segmented_data.csv', index=False)
print(f"✅ Successfully exported RFM metrics for {len(rfm_df)} unique customers!")
print("\n--- RFM Segmented Data Preview ---")

rfm_df.head()

✅ Successfully exported RFM metrics for 4338 unique customers!

--- RFM Segmented Data Preview ---


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment
0,14646,1,73,280206.02,5,5,5,555
1,18102,0,60,259657.30,5,5,5,555
2,17450,7,46,194390.79,5,5,5,555
3,16446,0,2,168472.50,5,3,5,535
4,14911,0,201,143711.17,5,5,5,555


In [16]:
rfm_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4338 entries, 0 to 4337
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CustomerID   4338 non-null   object 
 1   Recency      4338 non-null   int64  
 2   Frequency    4338 non-null   int64  
 3   Monetary     4338 non-null   float64
 4   R_Score      4338 non-null   int64  
 5   F_Score      4338 non-null   int64  
 6   M_Score      4338 non-null   int64  
 7   RFM_Segment  4338 non-null   object 
dtypes: float64(1), int64(5), object(2)
memory usage: 271.3+ KB


In [17]:
from google.colab import files
files.download('rfm_segmented_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
files.download('online_retail_clean.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>